# Whisper large-v3 LoRA fine-tune — Greek council ASR (dry-run)

**Purpose:** pipeline-validation + preliminary signal on the curated ~1.9k included
corrections (~1.8h). NOT proof of model gains (too little data — expect overfit).

**Runs on:** Kaggle (Settings → Accelerator **GPU T4 x2**, Internet **ON**). Use
**Save & Run All (Commit)** for a headless run — no keep-alive needed.

**Flow:** fetch `/api/export` (VPS) → build clips (decode each meeting mp3 once,
slice PCM) → LoRA fine-tune → WER on held-out val (corrections **and** a no-edit
regression set) → save adapter. Data source is already public; nothing to prep.


In [ ]:
# 1. Install (Kaggle has torch; add the rest)
!pip -q install -U transformers datasets peft accelerate evaluate jiwer faster-whisper librosa soundfile 2>/dev/null
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2. Config
EXPORT_URL = "https://79-76-114-184.sslip.io/api/export"   # VPS included-corrections export
MEETING_API = "https://opencouncil.gr/api/cities/{city}/meetings/{meeting}"
MODEL_ID   = "openai/whisper-large-v3"
LANGUAGE, TASK = "greek", "transcribe"
VAL_CITIES = {"orestiada", "argos"}        # held out from training (test = future data)
SAMPLE_N   = None                           # cap train rows for a fast smoke test, else None
VAL_REG_PER_MEETING = 8                      # no-edit utts sampled per val meeting (regression set)
SR = 16000
PAD_S = 0.2
MIN_DUR, MAX_DUR = 0.3, 30.0
SEED = 13
# LoRA (Codex: start small on tiny data)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR, EPOCHS, TRAIN_BS, GRAD_ACC = 1e-4, 4, 4, 2
OUT_DIR = "/kaggle/working/whisper-lora-greek"   # persisted on Commit; also copy to Drive if on Colab
import os, random, numpy as np
os.makedirs(OUT_DIR, exist_ok=True); random.seed(SEED); np.random.seed(SEED)

In [ ]:
# 3. Fetch the export (included corrections) + fetch meeting JSON helper
import requests, json, collections
def fetch_jsonl(url):
    r = requests.get(url, timeout=600); r.raise_for_status()
    return [json.loads(l) for l in r.text.splitlines() if l.strip()]
rows = fetch_jsonl(EXPORT_URL)
print("included rows:", len(rows))
by_city = collections.Counter(r["city_id"] for r in rows)
print("per city:", dict(by_city.most_common()))

def fetch_meeting(city, meeting):
    r = requests.get(MEETING_API.format(city=city, meeting=meeting),
                     headers={"User-Agent":"oc-ft/1.0","Accept":"application/json"}, timeout=120)
    r.raise_for_status(); return r.json()

In [ ]:
# 4. Audio: download each meeting mp3 once, decode to 16k mono PCM once, slice PCM
import librosa, hashlib, pathlib
CACHE = pathlib.Path("/kaggle/working/audio_cache"); CACHE.mkdir(exist_ok=True)
_pcm = {}
def _dl(url):
    p = CACHE / (hashlib.md5(url.encode()).hexdigest() + ".mp3")
    if not p.exists():
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(p, "wb") as f:
                for c in r.iter_content(1<<20): f.write(c)
    return p
def pcm_for(url):
    if url not in _pcm:
        y, _ = librosa.load(str(_dl(url)), sr=SR, mono=True)  # decode ONCE per meeting
        _pcm[url] = y
    return _pcm[url]
def clip(url, start, end):
    y = pcm_for(url)
    a = max(0, int((start - PAD_S) * SR)); b = min(len(y), int((end + PAD_S) * SR))
    return y[a:b]
def ok_span(start, end):
    d = (end or 0) - (start or 0); return MIN_DUR <= d <= MAX_DUR

In [ ]:
# 5. Build train / val-corrections / val-regression record lists
def rec(url, start, end, text):
    return {"audio": {"array": clip(url, start, end), "sampling_rate": SR}, "text": (text or "").strip()}

train_src = [r for r in rows if r["city_id"] not in VAL_CITIES and r.get("final_after_text") and ok_span(r["start"], r["end"])]
val_src   = [r for r in rows if r["city_id"] in VAL_CITIES and r.get("final_after_text") and ok_span(r["start"], r["end"])]
if SAMPLE_N: random.shuffle(train_src); train_src = train_src[:SAMPLE_N]

train_recs = [rec(r["audio_url"], r["start"], r["end"], r["final_after_text"]) for r in train_src]
valc_recs  = [rec(r["audio_url"], r["start"], r["end"], r["final_after_text"]) for r in val_src]

# Regression set: no-edit utterances from the val meetings (Codex: detect ordinary-speech degradation)
val_meetings = sorted({(r["city_id"], r["meeting_id"]) for r in val_src})
valr_recs = []
for city, mtg in val_meetings:
    try: mj = fetch_meeting(city, mtg)
    except Exception as e: print("skip", city, mtg, e); continue
    au = (mj.get("meeting") or {}).get("audioUrl")
    noedit = []
    for seg in mj.get("transcript") or []:
        for u in seg.get("utterances") or []:
            if u.get("lastModifiedBy") is None and ok_span(u.get("startTimestamp"), u.get("endTimestamp")) and (u.get("text") or "").strip():
                noedit.append(u)
    random.shuffle(noedit)
    for u in noedit[:VAL_REG_PER_MEETING]:
        valr_recs.append(rec(au, u["startTimestamp"], u["endTimestamp"], u["text"]))
print(f"train={len(train_recs)} val_corr={len(valc_recs)} val_reg={len(valr_recs)}")

In [ ]:
# 6. HF datasets + Whisper processor preprocessing
from datasets import Dataset
from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
def prep(b):
    a = b["audio"]
    b["input_features"] = processor.feature_extractor(a["array"], sampling_rate=a["sampling_rate"]).input_features[0]
    b["labels"] = processor.tokenizer(b["text"]).input_ids
    return b
ds_train = Dataset.from_list(train_recs).map(prep, remove_columns=["audio","text"])
ds_valc  = Dataset.from_list(valc_recs).map(prep, remove_columns=["audio","text"])
ds_valr  = Dataset.from_list(valr_recs).map(prep, remove_columns=["audio","text"]) if valr_recs else None

In [ ]:
# 7. Collator + metrics (raw WER, Greek-normalized WER, CER)
import torch, evaluate, unicodedata, re
from dataclasses import dataclass
@dataclass
class Collator:
    processor: object
    def __call__(self, feats):
        inp = [{"input_features": f["input_features"]} for f in feats]
        batch = self.processor.feature_extractor.pad(inp, return_tensors="pt")
        lab = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:,0] == self.processor.tokenizer.bos_token_id).all().cpu().item(): labels = labels[:,1:]
        batch["labels"] = labels; return batch
collator = Collator(processor)
wer_m, cer_m = evaluate.load("wer"), evaluate.load("cer")
_PUNCT = re.compile(r"[^\w\s]", re.UNICODE)
def gnorm(s):
    s = unicodedata.normalize("NFD", s.lower()); s = "".join(c for c in s if unicodedata.category(c)!="Mn")
    s = unicodedata.normalize("NFC", s).replace("ς","σ"); return re.sub(r"\s+"," ", _PUNCT.sub(" ", s)).strip()
def metrics(pred):
    ids = pred.predictions; lab = pred.label_ids
    lab = np.where(lab != -100, lab, processor.tokenizer.pad_token_id)
    P = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
    R = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    return {"wer": 100*wer_m.compute(predictions=P, references=R),
            "wer_norm": 100*wer_m.compute(predictions=[gnorm(x) for x in P], references=[gnorm(x) for x in R]),
            "cer": 100*cer_m.compute(predictions=P, references=R)}

In [ ]:
# 8. Load model, apply LoRA (freeze encoder), Seq2Seq trainer
import torch
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.config.suppress_tokens = []
model.generation_config.language, model.generation_config.task = LANGUAGE, TASK
model.model.encoder.requires_grad_(False)            # freeze encoder (domain-adapt decoder)
model.gradient_checkpointing_enable(); model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules=["q_proj","v_proj"]))
model.print_trainable_parameters()
args = Seq2SeqTrainingArguments(output_dir=OUT_DIR, per_device_train_batch_size=TRAIN_BS,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, warmup_ratio=0.1, num_train_epochs=EPOCHS,
    fp16=True, predict_with_generate=True, generation_max_length=225, eval_strategy="epoch",
    save_strategy="epoch", logging_steps=20, report_to=[], remove_unused_columns=False,
    label_names=["labels"], load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
    seed=SEED, per_device_eval_batch_size=8)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_valc,
                         data_collator=collator, compute_metrics=metrics, processing_class=processor)

In [ ]:
# 9. Baseline (pre-finetune) WER on val_corr + val_reg, then TRAIN
print("BASELINE val_corr:", trainer.evaluate(ds_valc))
if ds_valr: print("BASELINE val_reg:", trainer.evaluate(ds_valr))
trainer.train()
print("AFTER val_corr:", trainer.evaluate(ds_valc))
if ds_valr: print("AFTER val_reg (regression check):", trainer.evaluate(ds_valr))

In [ ]:
# 10. Save the LoRA adapter (persisted as Kaggle output on Commit)
model.save_pretrained(OUT_DIR); processor.save_pretrained(OUT_DIR)
import json
json.dump({"model": MODEL_ID, "lora_r": LORA_R, "lr": LR, "epochs": EPOCHS, "seed": SEED,
           "n_train": len(train_recs), "n_val_corr": len(valc_recs), "n_val_reg": len(valr_recs),
           "val_cities": sorted(VAL_CITIES)}, open(OUT_DIR+"/run_meta.json","w"), ensure_ascii=False, indent=2)
print("saved ->", OUT_DIR)
# On Colab instead of Kaggle: from google.colab import drive; drive.mount('/content/drive'); shutil.copytree(OUT_DIR, '/content/drive/MyDrive/whisper-lora-greek')

## 11. N-best / confidence experiment (optional)

faster-whisper gives per-segment `avg_logprob` / `no_speech_prob` and per-word
`probability`. We emit a **condensed JSON only for low-confidence spans** so a
downstream LLM corrects only uncertain text. ⚠️ Codex caveat: word/segment
probability ≠ true N-best hypotheses — for real alternatives use beam search with
`num_return_sequences` via `transformers.generate` (commented below).

In [ ]:
# 11. Confidence-gated condensed output (run on any meeting clip)
from faster_whisper import WhisperModel
CONF_AVG_LOGPROB, CONF_WORD_P = -1.0, 0.7
fw = WhisperModel("large-v3", device="cuda", compute_type="float16")
def annotate(wav_path):
    segs, _ = fw.transcribe(wav_path, language="el", beam_size=5, word_timestamps=True)
    out = []
    for s in segs:
        low = (s.avg_logprob < CONF_AVG_LOGPROB) or any((w.probability or 1) < CONF_WORD_P for w in (s.words or []))
        if low:
            out.append({"text": s.text.strip(), "conf": round(float(s.avg_logprob),2),
                        "low_words": [w.word for w in (s.words or []) if (w.probability or 1) < CONF_WORD_P]})
        else:
            out.append(s.text.strip())
    return out
# example: annotate("/kaggle/working/audio_cache/<hash>.mp3 sliced to a wav")  # feed an LLM the JSON
print("annotate() ready — feed its output to the fix-task LLM")

## Next
- Compare BASELINE vs AFTER on **val_corr** (did corrections improve?) and **val_reg**
  (did ordinary speech regress? — the key safety check).
- Scale up: add the **no-edit backbone** to training; sweep on the mini PC (Track 2).
- For a trustworthy number: ≥3 seeds, report mean/range, bootstrap CIs by meeting.